Process the review results into a single review.jsonl file so that it can be merged with other results. 

First match it back up to its source file. To do this we need to do some minor fixes. There was a small number of examples in v2024.04.11 where the claim.flat was updated but the claim object wasn't updated to match so their significance fields don't match. Since claim.flat was used for all testing, that is what we put into the review html file and that's what we should be propogating downstream

In [1]:
import pandas as pd
from nli.util import convert_col_to_words

def read_dataset(filename):
    # fix bad claim objects
    # claim.flat is correct but the claim object wasn't updated properly afte significance was flipped, do that now
    print(f"reading main datafile: {filename}")
    df = pd.read_json(filename, lines=True, dtype=False)
    df["document.pmid"] = (
        df["document.pmid"].apply(lambda x: "" if pd.isnull(x) else str(int(x))).astype(str)
    )
    df["_flat_sig"] = df["claim.flat"].str.extract(r"significance is ([^;]+)")
    df["_sig"] = df.claim.apply(
        lambda claims: [v for k, v in claims[0]["content"] if k == "significance"][0]
    )

    def replace_with_flat(row):
        unflat = [item.split(" is ") for item in row["claim.flat"].split(";")]
        assert len(unflat) == len(
            row.claim[0]["content"]
        ), f'the length of the flat claim ({len(unflat)}) should match the length of the built claim ({len(row.claim[0]["content"])})'

        for i in range(len(row.claim[0]["content"])):
            assert unflat[i][0].strip() == convert_col_to_words(
                row.claim[0]["content"][i][0]
            ), f'columns don\'t match from flattened ({[unflat[i][0].strip()]}) claim and built ({convert_col_to_words(row.claim[0]["content"][i][0])}) claim'
            row.claim[0]["content"][i][1] = unflat[i][1].strip()

    df[df._flat_sig != df._sig].apply(replace_with_flat, axis=1)
    total_examples = df.shape[0]
    print(f"read {total_examples} data examples")
    df = df.drop(columns=[c for c in df.columns if c.startswith("_")])
    return df


df = read_dataset('../../../builds/v2024.04.11/data.jsonl.gz')
df.head()

reading main datafile: ../../../builds/v2024.04.11/data.jsonl.gz
read 12761 data examples


,claim,claim.flat,claim.generated,claim.hash,claim.id,claim.source,conversation,display_id,document.doi,document.pmcid,...,gold_label_name,hypothesis.tag,hypothesis.username,id,in_table,is_flagged,original_example,partition,pmid.partition,reviews
0,"[{'content': [['molecularProfile', 'CBL V391I'...",molecular profile is CBL V391I; evidence type ...,False,3115,7297,civic,"[{'comment': '', 'created': '2022-03-09T04:33:...",b72b2d0b-6e93-41c7-9b32-c35d912a3aed,10.1038/s41598-017-09078-4,PMC5569108,...,SUPPORTS,eid:7297-ex:2,creisle,EID7297_ex:2_creisle_default,False,False,NaN,dev,dev,[]
1,"[{'content': [['molecularProfile', 'KLF4 K409Q...",molecular profile is KLF4 K409Q; evidence type...,False,2846,9557,civic,"[{'comment': '', 'created': '2022-03-08T04:33:...",14216a39-2114-44fd-a112-dc12475b1101,10.1186/s40478-020-00912-x,PMC7118946,...,SUPPORTS,eid:9557-ex:2,creisle,EID9557_ex:2_creisle_default,False,False,NaN,train,train,[]
2,"[{'content': [['molecularProfile', 'ETS2 RS461...",molecular profile is ETS2 RS461155; evidence t...,False,307,1060,civic,"[{'comment': '', 'created': '2021-12-29T21:05:...",43c162ac-181c-4260-aea2-4edf2cea7fad,10.18632/oncotarget.7433,PMC4941289,...,NEI,eid:1060-status:NEI,creisle,EID1060_ex:1_creisle_NEI,False,False,NaN,train,train,[]
3,"[{'content': [['molecularProfile', 'TP53 P47S'...",molecular profile is TP53 P47S; evidence type ...,False,1619,1209,civic,"[{'comment': '', 'created': '2023-02-01T01:07:...",05b84181-3841-4fb3-ac7d-353aca894673,10.1101/gad.275891.115,PMC4840298,...,SUPPORTS,eid:1209,epleasance,EID1209_ex:1_epleasance_default,False,False,NaN,train,train,[]
4,"[{'content': [['molecularProfile', 'MSH6 LOSS'...",molecular profile is MSH6 LOSS; evidence type ...,False,1198,1878,civic,"[{'comment': 'covers: gene, variant', 'created...",1e953da3-b86f-424e-af4c-4925e10d7543,10.1186/s40425-015-0104-y,PMC4678532,...,SUPPORTS,eid:1878-ex:2,creisle,EID1878_ex:2_creisle_default,False,False,NaN,test,test,[]


Now read in the review files. We want to create review objects which look something like this (example from previous review compilation)

```json
{
    "comment": "evidence is not predictive based on this variant",
    "group_review_comments": [],
    "group_review_label": "REFUTES",
    "group_review_round": "IAv2_error_review",
    "group_review_tags": [
        "Bot Generated",
        "Gold Label Incorrect"
    ],
    "group_review_validated": false,
    "label": "NEI",
    "original_label": "NEI",
    "reason": "",
    "round": "IAv2_civicfact2023.10.20",
    "username": "camgrisdale"
}
```

In [2]:
import glob

reviews = []

for filename in glob.glob('*.jsonl'):
    reviews.append(pd.read_json(filename, dtype=False, lines=True))
review_df = pd.concat(reviews)
review_df['username'] = 'creisle' # I was the only reviewer of the model errors file
review_df['round'] = 'model error review 2024.05.06'
review_df = review_df.rename(columns={'label': 'reason'})
review_df = review_df.explode('reason')
review_df = review_df[~review_df.reason.fillna('').str.startswith(r'[')]
review_df['label'] = review_df.reason.str.extract(r'^(\w+)\.?')
review_df['reason'] = review_df.reason.str.replace(r'^(\w+)(\.\s+)?', '', regex=True).fillna('')
review_df['original_label'] = review_df.gold_label_name
review_df = review_df[review_df.reason != '']
review_df.head()

,id,text,reviews,claim.hash,display_id,claim.source,document.pmid,evidence.hash,evidence.source,gold_label_name,...,hypothesis.username,in_table,is_flagged,original_example,partition,pmid.partition,username,round,label,original_label
0,EID4286_desc_none_NEI,"<div class=""example""><div class=""question""><h2...","[{'label': 'NEI', 'reason': 'The disease (Lung...",6048,469c7b07-5897-40dd-8187-187bf144b6fc,civic,24353160,10206,civic,NEI,...,NaN,NaN,NaN,NaN,NaN,NaN,creisle,model error review 2024.05.06,NEI,NEI
1,EID4294_desc_none_NEI,"<div class=""example""><div class=""question""><h2...","[{'label': 'NEI', 'reason': 'The disease (Lung...",7155,2e9de5b0-75f8-42d7-a7de-23cdd1e52f0e,civic,24893891,10212,civic,NEI,...,NaN,NaN,NaN,NaN,NaN,NaN,creisle,model error review 2024.05.06,NEI,NEI
0,EIDclaim:4397_ex:30096481p1s8_p1s2-p1s3_factbo...,"<div class=""example""><div class=""question""><h2...",[],4397,04756bd2-b7a5-47ca-9e54-6bcb467199c2,neutral-bot,30096481,4398,pubmed,SUPPORTS,...,NaN,NaN,NaN,NaN,NaN,NaN,creisle,model error review 2024.05.06,NEI,SUPPORTS
1,EID145_desc_none_NEI,"<div class=""example""><div class=""question""><h2...","[{'label': 'NEI', 'reason': 'The disease (Colo...",5671,c7037f32-322e-48a9-aa01-2d5d493df4ac,civic,20978259,8517,civic,NEI,...,NaN,NaN,NaN,NaN,NaN,NaN,creisle,model error review 2024.05.06,SUPPORTS,NEI
2,EID151_desc_none_NEI,"<div class=""example""><div class=""question""><h2...","[{'label': 'NEI', 'reason': 'The variant (NPM1...",5675,c60fb1b2-43c1-48f0-9070-2d8471d8cbe4,civic,24927407,8522,civic,NEI,...,NaN,NaN,NaN,NaN,NaN,NaN,creisle,model error review 2024.05.06,NEI,NEI


Now join back to the original dataset

In [3]:
review_df = review_df.groupby([
    'id', 'claim.hash', 'evidence.hash', 'display_id'
     ]).apply(lambda subset_df: subset_df[['label', 'original_label', 'username', 'round','reason']].to_dict('records')).reset_index(name='reviews')
review_df = review_df.merge(df.drop(columns=['reviews']))
review_df.head()

,id,claim.hash,evidence.hash,display_id,reviews,claim,claim.flat,claim.generated,claim.id,claim.source,...,example,generated,gold_label_name,hypothesis.tag,hypothesis.username,in_table,is_flagged,original_example,partition,pmid.partition
0,EID1062_desc_none_NEI,6177,9127,3e715f46-4d51-4509-ba76-d9feb33ff3df,"[{'label': 'SUPPORTS', 'original_label': 'NEI'...","[{'content': [['molecularProfile', 'FGF13 CYTO...",molecular profile is FGF13 CYTOPLASMIC EXPRESS...,False,1062,civic,...,description,False,NEI,,none,False,False,NaN,dev,dev
1,EID11206_desc_none_REFUTES,11632,11533,d7d90d8b-f9d6-4b71-9b98-a26fb028970e,"[{'label': 'REFUTES', 'original_label': 'REFUT...","[{'content': [['molecularProfile', 'PPP2R2A Mu...",molecular profile is PPP2R2A Mutation; evidenc...,False,11206,civic,...,description,False,REFUTES,,none,False,False,NaN,test,test
2,EID11218_desc_none_NEI,3492,11539,4b3ea43b-89a5-47d0-aba8-896ed1b1ac37,"[{'label': 'NEI', 'original_label': 'NEI', 'us...","[{'content': [['molecularProfile', 'ALK Fusion...",molecular profile is ALK Fusion; evidence type...,False,11218,civic,...,description,False,NEI,,none,False,False,NaN,dev,dev
3,EID11447_desc_none_NEI,8399,11559,645209a9-f519-46f1-b6e7-9117c82ec148,"[{'label': 'NEI', 'original_label': 'NEI', 'us...","[{'content': [['molecularProfile', 'ZFTA::RELA...",molecular profile is ZFTA::RELA Methylation Pr...,False,11447,civic,...,description,False,NEI,,none,False,False,NaN,test,test
4,EID11534_desc_none_NEI,11644,11684,7780160b-8d06-4c5d-bb68-252dca748995,"[{'label': 'NEI', 'original_label': 'NEI', 'us...","[{'content': [['molecularProfile', 'TFEB Fusio...",molecular profile is TFEB Fusion; evidence typ...,False,11534,civic,...,description,False,NEI,,none,False,False,NaN,test,test


finally fix the hashes to match the fixed claim objects

In [4]:
import json

from article_utils.db import connect, get_or_create_textid


review_df['_claim'] = review_df['claim'].apply(lambda claims: json.dumps([c['content'] for c in claims]))

conn = connect('../../articles_db.sqlite')
try:
    review_df['claim.hash'] = review_df._claim.apply(lambda claim_text: get_or_create_textid(conn, claim_text))
finally:
    conn.close()

now we need to update the gold_label_name for any were the majority review has changed the label

In [5]:
from statistics import mode

review_df['_gold_label_name'] = review_df.reviews.apply(lambda reviews: mode([r['label'] for r in reviews]))
review_df[review_df._gold_label_name != review_df.gold_label_name][['gold_label_name', '_gold_label_name']]

,gold_label_name,_gold_label_name
0,NEI,SUPPORTS
7,NEI,SUPPORTS
8,NEI,SUPPORTS
14,NEI,SUPPORTS
17,NEI,SUPPORTS
23,NEI,SUPPORTS
24,NEI,REFUTES
25,NEI,SUPPORTS
26,NEI,SUPPORTS
29,REFUTES,SUPPORTS


In [6]:
review_df['gold_label_name'] = review_df._gold_label_name
review_df = review_df.drop(columns=[c for c in review_df.columns if c.startswith('_')])
review_df.head()

,id,claim.hash,evidence.hash,display_id,reviews,claim,claim.flat,claim.generated,claim.id,claim.source,...,example,generated,gold_label_name,hypothesis.tag,hypothesis.username,in_table,is_flagged,original_example,partition,pmid.partition
0,EID1062_desc_none_NEI,6177,9127,3e715f46-4d51-4509-ba76-d9feb33ff3df,"[{'label': 'SUPPORTS', 'original_label': 'NEI'...","[{'content': [['molecularProfile', 'FGF13 CYTO...",molecular profile is FGF13 CYTOPLASMIC EXPRESS...,False,1062,civic,...,description,False,SUPPORTS,,none,False,False,NaN,dev,dev
1,EID11206_desc_none_REFUTES,11632,11533,d7d90d8b-f9d6-4b71-9b98-a26fb028970e,"[{'label': 'REFUTES', 'original_label': 'REFUT...","[{'content': [['molecularProfile', 'PPP2R2A Mu...",molecular profile is PPP2R2A Mutation; evidenc...,False,11206,civic,...,description,False,REFUTES,,none,False,False,NaN,test,test
2,EID11218_desc_none_NEI,3492,11539,4b3ea43b-89a5-47d0-aba8-896ed1b1ac37,"[{'label': 'NEI', 'original_label': 'NEI', 'us...","[{'content': [['molecularProfile', 'ALK Fusion...",molecular profile is ALK Fusion; evidence type...,False,11218,civic,...,description,False,NEI,,none,False,False,NaN,dev,dev
3,EID11447_desc_none_NEI,8399,11559,645209a9-f519-46f1-b6e7-9117c82ec148,"[{'label': 'NEI', 'original_label': 'NEI', 'us...","[{'content': [['molecularProfile', 'ZFTA::RELA...",molecular profile is ZFTA::RELA Methylation Pr...,False,11447,civic,...,description,False,NEI,,none,False,False,NaN,test,test
4,EID11534_desc_none_NEI,11644,11684,7780160b-8d06-4c5d-bb68-252dca748995,"[{'label': 'NEI', 'original_label': 'NEI', 'us...","[{'content': [['molecularProfile', 'TFEB Fusio...",molecular profile is TFEB Fusion; evidence typ...,False,11534,civic,...,description,False,NEI,,none,False,False,NaN,test,test


In [7]:
from article_utils.util import dump_jsonl

dump_jsonl(review_df, 'reviews.jsonl')